# Bonus Task
Mandelbrot Set

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def mandelbrot(c, max_iter=100):
    """Computes the number of iterations before divergence."""
    z = 0
    for n in range(max_iter):
        if abs(z) > 2:
            return n
        z = z*z + c
    return max_iter

def mandelbrot_set(width, height, x_min, x_max, y_min, y_max, max_iter=100):
    """Generates the Mandelbrot set image."""
    x_vals = np.linspace(x_min, x_max, width)
    y_vals = np.linspace(y_min, y_max, height)
    image = np.zeros((height, width))

    for i in range(height):
        for j in range(width):
            c = complex(x_vals[j], y_vals[i])
            image[i, j] = mandelbrot(c, max_iter)

    return image

# Parameters
width, height = 1000, 800
x_min, x_max, y_min, y_max = -2, 1, -1, 1

# Generate fractal
image = mandelbrot_set(width, height, x_min, x_max, y_min, y_max)

# Display
plt.imshow(image, cmap='inferno', extent=[x_min, x_max, y_min, y_max])
plt.colorbar()
plt.title("Mandelbrot Set")
plt.show()


## Task B.1
### Cython Optimization of the Mandelbrot Set

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mandelbrot_cy import mandelbrot_set  # Cython-compiled: typed mandelbrot + memoryviews

# Parameters
width, height = 1000, 800
x_min, x_max, y_min, y_max = -2, 1, -1, 1

# Generate fractal (Cython-optimized)
image = mandelbrot_set(width, height, x_min, x_max, y_min, y_max)

# Display
plt.imshow(image, cmap='inferno', extent=[x_min, x_max, y_min, y_max])
plt.colorbar()
plt.title("Mandelbrot Set (Cython)")
plt.show()


## Task B.2
### Vectorized NumPy and PyTorch GPU Mandelbrot

Instead of nested loops, use a 2D complex grid and element-wise operations with Boolean masking to track which points have diverged. PyTorch version runs on GPU.

In [ ]:
import torch
import matplotlib.pyplot as plt


def mandelbrot_set_torch_gpu(width, height, x_min, x_max, y_min, y_max, max_iter=100, device=None):
    """PyTorch Mandelbrot on GPU: same vectorized logic with Boolean masking, all on device."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

    x_vals = torch.linspace(x_min, x_max, width, dtype=torch.float64, device=device)
    y_vals = torch.linspace(y_min, y_max, height, dtype=torch.float64, device=device)
    x_2d, y_2d = torch.meshgrid(x_vals, y_vals, indexing='xy')
    C = torch.complex(x_2d, y_2d)

    Z = torch.zeros_like(C, device=device)
    output = torch.full((height, width), max_iter, dtype=torch.float64, device=device)
    mask = torch.ones((height, width), dtype=torch.bool, device=device)

    for n in range(max_iter):
        Z[mask] = Z[mask] * Z[mask] + C[mask]
        diverged = torch.abs(Z) > 2
        just_diverged = diverged & mask
        output[just_diverged] = n
        mask &= ~diverged

    return output.cpu().numpy()


# Same parameters
width, height = 1000, 800
x_min, x_max, y_min, y_max = -2, 1, -1, 1

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
image_gpu = mandelbrot_set_torch_gpu(width, height, x_min, x_max, y_min, y_max, max_iter=100, device=device)
print(f"Computed on: {device}")

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
im = ax.imshow(image_gpu, cmap='inferno', extent=[x_min, x_max, y_min, y_max])
plt.colorbar(im, ax=ax)
ax.set_title("Mandelbrot Set (PyTorch vectorized, GPU)")
plt.tight_layout()
plt.show()

## Task B.3
### Execution time comparison and optimization comments

Compare baseline (nested loop), Cython, and vectorized versions (NumPy on CPU, PyTorch on CPU and GPU). Comment on the different optimizations and their performance.

In [ ]:
import numpy as np
import time
import torch
from mandelbrot_cy import mandelbrot_set as mandelbrot_set_cython


# --- Baseline: nested loop, Python complex (from Task B.1 intro) ---
def mandelbrot_baseline(c, max_iter=100):
    z = 0
    for n in range(max_iter):
        if abs(z) > 2:
            return n
        z = z * z + c
    return max_iter

def mandelbrot_set_baseline(width, height, x_min, x_max, y_min, y_max, max_iter=100):
    x_vals = np.linspace(x_min, x_max, width)
    y_vals = np.linspace(y_min, y_max, height)
    image = np.zeros((height, width))
    for i in range(height):
        for j in range(width):
            c = complex(x_vals[j], y_vals[i])
            image[i, j] = mandelbrot_baseline(c, max_iter)
    return image


# --- Vectorized NumPy (from Task B.2): 2D grid + Boolean masking, CPU ---
def mandelbrot_set_vectorized(width, height, x_min, x_max, y_min, y_max, max_iter=100):
    x_vals = np.linspace(x_min, x_max, width, dtype=np.float64)
    y_vals = np.linspace(y_min, y_max, height, dtype=np.float64)
    x_2d, y_2d = np.meshgrid(x_vals, y_vals)
    C = x_2d + 1j * y_2d
    Z = np.zeros_like(C)
    output = np.full((height, width), max_iter, dtype=np.float64)
    mask = np.ones((height, width), dtype=bool)
    for n in range(max_iter):
        Z[mask] = Z[mask] * Z[mask] + C[mask]
        diverged = np.abs(Z) > 2
        output[diverged & mask] = n
        mask &= ~diverged
    return output


# --- PyTorch vectorized: CPU and GPU ---
def mandelbrot_set_torch(width, height, x_min, x_max, y_min, y_max, max_iter=100, device=None):
    if device is None:
        device = torch.device('cpu')
    x_vals = torch.linspace(x_min, x_max, width, dtype=torch.float64, device=device)
    y_vals = torch.linspace(y_min, y_max, height, dtype=torch.float64, device=device)
    x_2d, y_2d = torch.meshgrid(x_vals, y_vals, indexing='xy')
    C = torch.complex(x_2d, y_2d)
    Z = torch.zeros_like(C, device=device)
    output = torch.full((height, width), max_iter, dtype=torch.float64, device=device)
    mask = torch.ones((height, width), dtype=torch.bool, device=device)
    for n in range(max_iter):
        Z[mask] = Z[mask] * Z[mask] + C[mask]
        diverged = torch.abs(Z) > 2
        output[diverged & mask] = n
        mask &= ~diverged
    if device.type == 'cuda':
        torch.cuda.synchronize()
    return output.cpu().numpy()


# --- Parameters and timing ---
width, height = 1000, 800
x_min, x_max, y_min, y_max = -2, 1, -1, 1
max_iter = 100
n_runs = 5  # number of runs per method for mean time

def time_fn(fn, *args, **kwargs):
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn(*args, **kwargs)
        times.append(time.perf_counter() - t0)
    return np.mean(times) * 1000  # ms

results = []

# Baseline (Python nested loop)
t_baseline = time_fn(mandelbrot_set_baseline, width, height, x_min, x_max, y_min, y_max, max_iter)
results.append(("Baseline (nested loop, Python)", t_baseline))

# Cython (typed + memoryviews, no GIL in inner loop)
t_cython = time_fn(mandelbrot_set_cython, width, height, x_min, x_max, y_min, y_max, max_iter)
results.append(("Cython (typed + memoryviews)", t_cython))

# NumPy vectorized (CPU)
t_np_vec = time_fn(mandelbrot_set_vectorized, width, height, x_min, x_max, y_min, y_max, max_iter)
results.append(("NumPy vectorized (CPU)", t_np_vec))

# PyTorch vectorized CPU
t_torch_cpu = time_fn(mandelbrot_set_torch, width, height, x_min, x_max, y_min, y_max, max_iter, torch.device('cpu'))
results.append(("PyTorch vectorized (CPU)", t_torch_cpu))

# PyTorch vectorized GPU (if available)
gpu_device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else None)
if gpu_device is not None:
    torch.cuda.synchronize() if gpu_device.type == 'cuda' else None
    t_torch_gpu = time_fn(mandelbrot_set_torch, width, height, x_min, x_max, y_min, y_max, max_iter, gpu_device)
    if gpu_device.type == 'cuda':
        torch.cuda.synchronize()
    results.append((f"PyTorch vectorized ({gpu_device.type.upper()})", t_torch_gpu))
else:
    results.append(("PyTorch vectorized (GPU)", "N/A (no CUDA/MPS)"))

# Print table
print("Execution time (mean over {} runs, {}x{} pixels, max_iter={})\n".format(n_runs, width, height, max_iter))
print("{:40s} {:>12s}".format("Method", "Time (ms)"))
print("-" * 54)
for name, t in results:
    print("{:40s} {:>12}".format(name, f"{t:.2f}" if isinstance(t, (int, float)) else t))

### Comments on optimizations and performance

- **Baseline (nested loop, Python):** Slowest, because every pixel does a Python loop and uses Python `complex` and `abs()`; each iteration has interpreter and type overhead. Good as a reference.

- **Cython (typed + memoryviews):** Much faster than baseline. Types (`double`, `int`) and typed memoryviews remove Python overhead in the inner loop; the core iteration compiles to C. `nogil` and no bounds checking further reduce cost. Often comparable to or faster than NumPy vectorized on CPU for this workload because there is no per-iteration Python masking and no redundant work on already-diverged pixels.

- **NumPy vectorized (CPU):** Replaces pixel loops with array ops and Boolean masking. Fast due to contiguous memory and optimized BLAS/vectorized ops, but each iteration updates and masks the full grid (including already-diverged points), so it does more total work per iteration than Cython. Typically faster than baseline, but may be slower than Cython at high resolution or when many points diverge early.

- **PyTorch vectorized (CPU):** Same algorithm as NumPy vectorized but on `torch` tensors. CPU runtime is usually in the same ballpark as NumPy (sometimes a bit slower due to framework overhead unless batch sizes are large).

- **PyTorch vectorized (GPU):** Same vectorized logic on GPU (CUDA or MPS). For large grids, parallelism and high memory bandwidth can make it the fastest. For small grids, transfer and kernel launch overhead can make GPU slower than a well-optimized CPU (Cython or NumPy). Speedup over CPU grows as `width × height` and `max_iter` increase.